# Lab 4: the reviewer in the pipeline

Lab 4 costs: nothing in this notebook. On GitHub, about $0.25 across three review runs if
you authenticate with an API key, or nothing beyond your plan if you authenticate with a
Claude subscription, where the runs count against your usage rather than a bill. Section 4
is where you choose, and most people should choose the subscription.

Scenario 5, Claude Code for continuous integration. You take a copy of a small Python
repository, ask Claude Code for a change on a branch, put the reviewing action on the
repository, and open the pull request that sets it off.

Nothing in this notebook runs. Every step is a command in a terminal or a button on
GitHub, and the notebook is the script you follow. Four steps, in this order:

| Step | Where |
|---|---|
| 1. Take your own copy of `tictactoe` | GitHub |
| 2. Ask Claude Code for a change, on a branch | Your terminal |
| 3. Install the action | Your terminal, then GitHub |
| 4. Open the pull request and read the review | GitHub |

The repository is `github.com/puria-izady/tictactoe`: noughts and crosses in the terminal,
four modules of a few dozen lines, a full test suite and an opponent that never loses. It
is small enough to read before the review comments on it, which is what makes the comments
worth arguing with.

## 1. Take your own copy

Open `github.com/puria-izady/tictactoe` and press the green **Use this template** button,
then **Create a new repository**. Make it public or private, either works.

**Not Fork.** On a public repository GitHub withholds secrets from any run triggered by a
fork's pull request, so the review would start and then find it has no key. This catches
people, and it catches them after the workflow has already been written.

Then clone it and check the tests pass before you change anything:

```bash
git clone https://github.com/YOUR-NAME/tictactoe
cd tictactoe
uv run --with pytest pytest -q
```

## 2. What the copy already tells a reviewer

Two files came with the template, and they are the reason this lab is about review quality
rather than about YAML. Open them now, before the reviewer reads them.

**`CLAUDE.md`** holds the conventions. A review in continuous integration has nobody to
ask: whatever it knows about this repository, it knows because the repository told it.
Three of its entries are load bearing here, and all three license something that looks like
a defect. `place()` copies the board on every call rather than mutating it. `_score` is
cached with no bound. Only `cli.py` prints, takes input, or touches a file. A reviewer that
does not know these has already been argued about will file all three, and be wrong three
times.

**`.claude/rules/review-criteria.md`** holds the policy. "Flag any issues you find" is the
instruction that produces the noise: it names no category, so every observation qualifies,
and it anchors no severity, so the same defect comes back high on Monday and low on
Thursday. Four parts of the file replace that, and each one prevents a specific failure.

| Part of the file | The failure it prevents |
|---|---|
| Reportable categories | A true observation outside the four categories arriving as a finding |
| The do-not-flag list | The three licensed conventions being reported as defects |
| Severity anchored to a code example | "I am not sure" being filed as a low rather than not filed |
| The `category:path:symbol` id recipe | The same defect coming back as a new comment after a rebase |

It lives in `.claude/rules/` rather than inside `CLAUDE.md` because a review policy is
argued about and edited on its own, and a rules file can carry a `paths:` block so it loads
for the files it governs rather than for every prompt in the repository:

```yaml
---
paths: ["tictactoe/**/*.py", "tests/**/*.py"]
---
```

`CLAUDE.md` pulls it in with one line at the bottom, `@.claude/rules/review-criteria.md`.
That line matters more than it looks, and section 6 takes it out to show what it was doing.

One change worth knowing about while you are here: `--bare` skips discovery of hooks,
skills, commands, subagents, plugins, MCP servers and `CLAUDE.md`, and it is documented as
becoming the default for `-p` in a future release. That would quietly take both of these
files away from every run that relies on them.

## 3. Ask for the change, on a branch

Now give the repository something worth reviewing. Make the branch first, then run Claude
Code in the clone and paste the prompt below as it stands:

```bash
git switch -c feature/game-log
claude
```

> Add game logging to this package. When a game finishes, append one line to a log file
> recording the outcome, the cells that were played in order, and the name of the person
> playing. Add a `--log PATH` option for where the file goes, defaulting to
> `~/.tictactoe/games.log`, and a `--player NAME` option for the name that gets recorded.
> What the game prints on screen must not change. Commit it in one commit when the tests
> pass.

It is a small feature and a deliberate one. It adds the first code in this package that
writes to a file, which gives the review something real in three of its four categories: a
path to expand and a directory that may not exist, a person's name heading towards a file
that gets shared, and at least one new branch that a test has to reach.

Do not push yet. Read the diff first, so that when the comments arrive you are comparing
them against a change you understand rather than against a surprise:

```bash
git show --stat
git diff main
```

## 4. Install the action

Three things have to be in place, and all three exist to serve the one file at the bottom
of this section: an app that may comment on your pull requests, a credential the job can
authenticate with, and the workflow that starts the review.

**One. Install the Claude GitHub App.** Open `github.com/apps/claude` and install it on
your copy of the repository. The action relies on three of the app's permissions, all read
and write: contents, issues, and pull requests. This is where the inline comments come from.
It is also why the workflow below asks for so little: the app posts, not the workflow token.

**Two. Store a credential as a repository secret.** The job has to authenticate as you,
and there are two ways to let it. They are not interchangeable in the workflow file, so read
the table before you pick one.

| | On a Claude subscription | On an API key |
|---|---|---|
| Secret name | `CLAUDE_CODE_OAUTH_TOKEN` | `ANTHROPIC_API_KEY` |
| Workflow input | `claude_code_oauth_token` | `anthropic_api_key` |
| Where the value comes from | `claude setup-token` | The Claude Console |
| What the runs cost | Counted against your plan's usage | Billed per run |

**Most students want the first column.** If you already run Claude Code on a Pro, Max, Team
or Enterprise plan, you have everything you need and the review runs cost you nothing
further. Generate the token from the clone:

```bash
claude setup-token
```

It prints a long-lived OAuth token to the terminal. Three things about that token, and all
three catch somebody:

- **It is printed and not saved.** Nothing writes it to disk, so copy it out of the terminal
  when it appears. If you lose it, run the command again and get a new one.
- **It is tied to you**, to the subscription of whoever ran the command. That is fine for
  your own copy of a teaching repository. It is the reason a shared team or organisation
  secret should be an API key instead: an OAuth token leaves with the person.
- **It expires.** When the workflow starts failing to authenticate months from now, nothing
  is broken. Generate a fresh token and overwrite the secret.

Store it on the repository:

```bash
gh secret set CLAUDE_CODE_OAUTH_TOKEN
```

`gh` prompts for the value and never puts it in your shell history. Paste the token and
press return.

On an API key instead, the same command with the other name, and paste a key from the Claude
Console:

```bash
gh secret set ANTHROPIC_API_KEY
```

Either way, check it landed. This prints the name and the date, never the value:

```bash
gh secret list
```

**Three. Add the workflow**, on `main`, at `.github/workflows/claude-review.yml`. This file
is the lab. It is the official review workflow, unedited:

```yaml
name: Code Review
on:
  pull_request:
    types: [opened, synchronize, ready_for_review, reopened]
jobs:
  review:
    runs-on: ubuntu-latest
    permissions:
      contents: read
      pull-requests: read
      issues: read
      id-token: write
    steps:
      - uses: actions/checkout@v6
        with:
          fetch-depth: 1
      - uses: anthropics/claude-code-action@v1
        with:
          anthropic_api_key: ${{ secrets.ANTHROPIC_API_KEY }}
          plugin_marketplaces: "https://github.com/anthropics/claude-code.git"
          plugins: "code-review@claude-code-plugins"
          prompt: "/code-review:code-review --comment ${{ github.repository }}/pull/${{ github.event.pull_request.number }}"
          claude_args: '--allowedTools "mcp__github_inline_comment__create_inline_comment"'
```

**If you took the subscription path, change one line of that file**, the credential, and
nothing else:

```yaml
          claude_code_oauth_token: ${{ secrets.CLAUDE_CODE_OAUTH_TOKEN }}
```

This is the single most common way to get a red workflow on the first try. The secret and
the input have to be the matching pair from the table above: an `ANTHROPIC_API_KEY` secret
read by a `claude_code_oauth_token` input is simply empty, and the run fails to
authenticate rather than telling you the names disagree.

```bash
git switch main
git add .github/workflows/claude-review.yml
git commit -m "Review on every pull request"
git push
```

Four lines in that file decide how it behaves.

| Line | What it decides |
|---|---|
| `on: pull_request` with its four types | When a review happens: a new pull request, a new push to one, a draft marked ready, a closed one reopened |
| `plugin_marketplaces` and `plugins` | Where the review logic comes from, rather than a prompt you maintain |
| `prompt`, and the `--comment` in it | Without `--comment` the review still runs and still bills, and the findings go to the workflow run log where nobody reads them |
| `claude_args` | The one tool that may run |

That last one is the line people leave out, and the docs tell you to keep it even though the
skill's own `allowed-tools` frontmatter names the same tool. The action starts the server
that posts inline comments only when `--allowedTools` in `claude_args` names it, so without
`mcp__github_inline_comment__create_inline_comment` the review runs, bills you, and posts
nothing.

The permissions are read only on the pull request, which looks wrong until you remember step
one: the app you installed posts the comments. `id-token: write` is what lets the action
prove who it is.

One sentence on the shortcut, because it is worth knowing it exists rather than using it
here: `/install-github-app`, run inside Claude Code with repository admin, does all three of
these steps for you and opens a pull request carrying the workflow files. This lab does it
by hand because the file is the thing being taught, and because a generated file you then
edit is two procedures where one will do.

## 5. Open the pull request

Bring the branch up to date with `main`, which now carries the workflow, then push it and
open the pull request:

```bash
git switch feature/game-log
git merge main
git push -u origin feature/game-log
gh pr create --fill
gh run watch
```

The review posts within a couple of minutes. Read it against section 2 rather than against
your own taste, with four questions:

> Is every comment in one of the four reportable categories?

> Did any of the three licensed conventions come back as a finding?

> Is every comment carrying a fix that could be applied as written?

> Did it notice a branch the change added that no test reaches?

Then fix one of the findings and push again. `synchronize` is in the trigger list, so the
same pull request is looked at a second time, and what you are watching for is what does
**not** happen: the finding you fixed does not reappear, and the ones you left alone are not
filed again as though they were new. That is what the id recipe in the criteria file buys.
An id built from a line number changes the moment anything above it changes, and a finding
whose identity moves comes back as new on the next commit.

It is also worth knowing what the review declines to look at, because a quiet second run is
not necessarily a broken one. It skips draft and closed pull requests, pull requests it
judges too trivial or too automated to be worth a review, and pull requests that already
carry a comment from it. So if the second run posts nothing at all, read the run log before
you go looking for a fault in the workflow.

## 6. Take the rules away

The last step is the one that shows what any of this was worth, and it wants a pull request
of its own rather than another push to the first one. A second pull request is a clean
comparison, and it sidesteps the skip list at the end of section 5.

Branch from your merged change, comment out the last line of `CLAUDE.md`, and open a second
pull request carrying a small further change to the logging code:

```bash
git switch -c experiment/no-rules
# comment out the last line of CLAUDE.md:
#     # @.claude/rules/review-criteria.md
git commit -am "Review without the criteria file"
git push -u origin experiment/no-rules
gh pr create --fill
```

Watch the same repository get reviewed with the rules file gone.

The licensed conventions are the thing to watch. With `CLAUDE.md` no longer pointing at the
criteria, flagging the unbounded cache or the board copy is a perfectly reasonable thing for
a reviewer to do. The comments are true. They are still noise, because the team has already
had that argument and settled it, and a category that is wrong more often than it is right
teaches everyone to skim past the findings that are right.

Close that pull request without merging it when you have read the comments.

## What you built

Read this back against the diagram.

| Diagram | Where it happened |
|---|---|
| Pull request, changed files | Sections 3 and 5, your own change |
| CLAUDE.md | Section 2, shipped with the template |
| Review rules | Section 2, `.claude/rules/review-criteria.md` |
| Claude Code headless | Section 4, inside the action |
| Changed file extraction | Section 4, the action's own checkout |
| Review and test generation | Section 2, the testing category and the boundary rule |
| PR feedback | Section 5 |
| Stable finding identity | Section 2, the id recipe; section 5, the second run |
| Rerun without duplicates | Section 5 |
| Precision, recall, false positives | Section 5's four questions, and section 6's answer to them |

Three boxes on the diagram are taught in Chapter 14 and not practised here: the JSON finding
schema, the JSON finding and the validate step. Those belong to a different run, one that
wants the findings as data for a dashboard or a gate rather than as comments on a pull
request, and it takes `--output-format json` with `--json-schema`. Lesson 14.2 has it, and
the thing to remember from it is that a schema the CLI cannot use degrades silently: the run
succeeds, the exit code is 0, and `structured_output` is simply `null`. Check the payload,
never the exit code.

The decision to carry out of this lab: when a reviewer is reporting things nobody wants to
read, the fix is a written rule about what not to report, not a gentler adjective in the
prompt.